In [2]:
# ! pip install langchain --upgrade

In [3]:
# !pip install langchain langchain-community --upgrade

In [5]:
# imports
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_anthropic import ChatAnthropic
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv
import os

load_dotenv()

C:\Users\HP\AppData\Local\Temp\ipykernel_31152\420426530.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


True

In [31]:
# setup
llm = ChatAnthropic(model="claude-haiku-4-5-20251001")

# embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
# pip install -qU chromadb langchain-chroma
# from langchain_chroma import Chroma


In [33]:
# loading foodprice collection as langchain retriever
food_vectorstore  = Chroma(
    client  = chromadb.PersistentClient("../vectorstore/food_poverty_db/"),
    collection_name = "food_prices",
    embedding_function = embeddings
)

# loading poverty collection as langchain retriever
poverty_vectorstore = Chroma(
    client  = chromadb.PersistentClient("../vectorstore/food_poverty_db/"),
    collection_name = "poverty_mpi",
    embedding_function = embeddings
)

food_retriever = food_vectorstore.as_retriever(search_kwargs = {"k":9})
poverty_retriever = poverty_vectorstore.as_retriever(search_kwargs={"k": 9})


In [34]:
# conversation history setup
chat_history =[]

def retrieve_context(question):
    food_docs = food_retriever.invoke(question)
    poverty_docs = poverty_retriever.invoke(question)
    
    all_docs = food_docs + poverty_docs
    context = "\n".join([doc.page_content for doc in all_docs])
    return context

In [35]:
# main ask question
def ask(question):
    # if follow-up question, enrich it with last answer context
    if chat_history:
        last_answer = chat_history[-1].content
        enriched_question = f"{question} (Previous context: {last_answer[:200]})"
    else:
        enriched_question = question
    context= retrieve_context(enriched_question)
    
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", """You are a humanitarian data analyst assistant.
Answer questions based only on the provided context.
Be precise with numbers and facts.
Always cite specific figures when available.

Context: {context}"""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{question}")
            
        ]
    )
    # building chain
    chain = prompt | llm
    
    response = chain.invoke({
        "context" : context,
        "chat_history" : chat_history,
        "question": question
    })
    # updating history
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response.content))
    
    return response.content

## Sample questions

In [36]:
# test conversational memory
print(ask("Which state has the highest poverty in India?"))
print()
print(ask("How do food prices look in that state?")) # this has to find state name by its memory.

Based on the provided context, **Uttar Pradesh** has the highest poverty rate in India among the states mentioned.

Uttar Pradesh shows the highest poverty figures at:
- **68.79%** of population lives in poverty (with an MPI score of 0.3611)

This is significantly higher than other states in the data:
- Haryana: 10.7% and 7.17%
- Himachal Pradesh: 30.93% and 8.12%
- Karnataka: 16.82%
- Kerala: 13.3%
- Uttar Pradesh also has another data point at 40.69% and 22.94%

The 68.79% figure represents the highest poverty rate in the provided dataset.

Based on the provided context, food prices in Uttar Pradesh show the following for rice:

**Rice Retail Prices in Uttar Pradesh:**
- **8.0 INR per KG** throughout 2002 (consistent across January, February, March, April, May, June, July, and September 2002)
- **9.0 INR per KG** in December 2004

The data shows rice prices remained stable at 8.0 INR per KG throughout 2002, with a slight increase to 9.0 INR per KG by late 2004 (a 12.5% increase over 

In [37]:
print(ask("What is the poverty situation in Bihar?"))

Based on the provided context, **Bihar has a severe poverty situation** with multiple poverty measures:

**Poverty Rates in Bihar:**
- **77.37%** of population lives in poverty (with an MPI score of 0.4484) - the highest measure
- **52.41%** of population lives in poverty (with an MPI score of 0.2475)
- **34.66%** of population lives in poverty (with an MPI score of 0.1544)

**Key Observations:**

1. The highest poverty figure of 77.37% indicates that more than three-quarters of Bihar's population lives in poverty, which is extremely high.

2. The MPI (Multidimensional Poverty Index) score of 0.4484 associated with the highest poverty rate suggests significant deprivation across multiple dimensions beyond just income.

3. Compared to other states in the data:
   - Himachal Pradesh: 8.12% to 30.93%
   - Haryana: 7.17% to 39.04%
   
   Bihar's poverty rates are substantially higher than these states.

Bihar faces one of the most severe poverty challenges among the states mentioned in the

In [38]:
print(ask("Which state has the highest poverty in India?"))


Based on the provided context, **Bihar** has the highest poverty in India among the states mentioned.

Bihar shows the highest poverty rate at:
- **77.37%** of population lives in poverty (with an MPI score of 0.4484)

This is significantly higher than all other states in the data:
- Uttar Pradesh: 68.79% and 22.94%
- Himachal Pradesh: 30.93% and 8.12%
- Haryana: 10.7% and 7.17%

Bihar's 77.37% poverty rate means more than three-quarters of the population lives in poverty, making it the state with the highest poverty among those included in the provided context.


In [39]:
print(ask("Compare poverty between Kerala and Jharkhand"))

Based on the provided context, here is a comparison of poverty between **Kerala and Jharkhand**:

**Kerala - Poverty Rates:**
- 13.3% of population lives in poverty (MPI score: 0.0528)
- 1.07% of population lives in poverty (MPI score: 0.004)
- 0.86% of population lives in poverty (MPI score: 0.0031)

**Jharkhand - Poverty Rates:**
- 74.86% of population lives in poverty (MPI score: 0.4291)
- 46.5% of population lives in poverty (MPI score: 0.208)
- 30.6% of population lives in poverty (MPI score: 0.1318)

**Key Comparison:**

Jharkhand has **significantly higher poverty than Kerala** across all measures:

- Jharkhand's highest poverty rate (74.86%) is nearly **6 times higher** than Kerala's highest rate (13.3%)
- Jharkhand's MPI scores (0.1318 to 0.4291) are substantially higher, indicating greater multidimensional deprivation
- Kerala's lowest poverty rate is only 0.86%, compared to Jharkhand's lowest at 30.6%

Kerala demonstrates much better poverty outcomes and living conditions co

In [40]:
print(ask("What is the poverty situation in Bihar?"))
print()
print(ask("How have rice prices changed there over the years?"))
print()
print(ask("Compare this to a wealthier state like Kerala"))

Based on the provided context, **Bihar has a severe poverty situation** with multiple poverty measures:

**Poverty Rates in Bihar:**
- **77.37%** of population lives in poverty (with an MPI score of 0.4484) - the highest measure
- **52.41%** of population lives in poverty (with an MPI score of 0.2475)
- **34.66%** of population lives in poverty (with an MPI score of 0.1544)

**Key Observations:**

1. The highest poverty figure of 77.37% indicates that more than three-quarters of Bihar's population lives in poverty, which is extremely high.

2. The MPI (Multidimensional Poverty Index) score of 0.4484 associated with the highest poverty rate suggests significant deprivation across multiple dimensions beyond just income.

3. Compared to other states in the data:
   - Kerala: 0.86% to 13.3%
   - Haryana: 7.17% to 39.04%
   - Jharkhand: 30.6% to 74.86%

Bihar's poverty rates are among the highest, second only to Jharkhand's 74.86%.

Bihar faces one of the most severe poverty challenges amon

# Below is manual way of doing same thing,    I had to use langchain as manual way was not getting lost in 2nd question, 

In [ ]:
# # Load chromadb
# chroma_client = chromadb.PersistentClient("../vectorstore/food_poverty_db/")
# collection_foodprice = chroma_client.get_collection(name='food_prices')
# collection_poverty = chroma_client.get_collection(name='poverty_mpi')

# embed_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# # building retriever function -that takes a question, searches ChromaDB
# # return top 3 relevant documents from each collection and combining them

# def retrieve(question, n_results=10):
#     question_embedding = embed_model.encode(question).tolist()
    
#     food_docs = collection_foodprice.query(
#         query_embeddings = [question_embedding],
#         n_results = n_results
#         )['documents'][0]
#     poverty_docs = collection_poverty.query(
#         query_embeddings = [question_embedding],
#         n_results = n_results
#         )['documents'][0]
    
#     return food_docs + poverty_docs

In [ ]:
# conversation_history = []

# def ask(question):
#     docs = retrieve(question)
#     context_text = '\n'.join(docs)
    
#     prompt = f"""Context: {context_text}
#     Question: {question}
#     Anwser based only on context provided."""
    
#     conversation_history.append({'role': 'user', 'content': prompt})
    
#     response = anthropic_client.messages.create(
#         model = "claude-haiku-4-5-20251001",
#         max_tokens=1024,
#         system = """You are a humanitarian data analyst assistant. 
#             Answer questions based only on the provided context. 
#             Be precise with numbers and facts. 
#             Always cite specific figures when available.
#             """,
#         messages=[{'role': 'user', 'content': prompt}]
#     )
#     answer = response.content[0].text
#     conversation_history.append({'role': 'assistant', 'content': answer})
#     return answer


In [ ]:
# # test conversational memory
# print(ask("Which state has the highest poverty in India?"))
# print()
# print(ask("How do food prices look in that state?")) # this has to find state name by its memory.

Based on the context provided, **Uttar Pradesh has the highest poverty in India**, with **68.79% of the population living in poverty** and an MPI (Multidimensional Poverty Index) score of 0.3611.

This is the highest poverty rate among all the states mentioned in the provided data.

I'd be happy to help, but I need clarification on which state you're referring to. The context provided includes data for multiple states.

Based on the available food price data (Tomato retail prices), here's what I can share:

**States with Tomato Price Data:**

- **Assam**: Prices ranged from 40.0 INR/KG (2013) to 70.0 INR/KG (2020), showing an increase over time
- **Goa**: 50.0 INR/KG (2014)
- **Puducherry**: Prices decreased significantly from 20.0 INR/KG (2015) to 8.0 INR/KG (2023)

Could you please specify which state you'd like me to focus on? That way I can provide you with the most relevant food price information from the context provided.
